# GetAround - Analyse & Deployment

**Objectif** : analyser les retards de checkout et leur impact, puis construire un modele de pricing deploye via API.

- **Partie 1** : EDA sur les retards (delay analysis)
- **Partie 2** : ML pour le pricing (regression)

## Imports

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

---
# Partie 1 : Analyse des retards

## 1.1 Chargement des donnees

In [ ]:
df_delay = pd.read_excel('data/get_around_delay_analysis.xlsx')
df_delay.shape

In [ ]:
df_delay.head()

In [ ]:
df_delay.info()

In [ ]:
df_delay.describe()

## 1.2 Donnees manquantes

In [ ]:
df_delay.isnull().sum()

In [ ]:
# delay_at_checkout = NaN pour les locations annulees (pas de checkout)
# previous_ended_rental_id et time_delta = NaN quand pas de location precedente sur la meme voiture
print(f"Locations sans checkout (canceled): {df_delay[df_delay['state']=='canceled'].shape[0]}")
print(f"Locations sans precedent: {df_delay['previous_ended_rental_id'].isna().sum()}")

## 1.3 Vue d'ensemble

In [ ]:
total = len(df_delay)
ended = df_delay[df_delay['state'] == 'ended']
canceled = df_delay[df_delay['state'] == 'canceled']

print(f"Total: {total}")
print(f"Terminees: {len(ended)} ({len(ended)/total*100:.1f}%)")
print(f"Annulees: {len(canceled)} ({len(canceled)/total*100:.1f}%)")

In [ ]:
fig = px.pie(df_delay, names='checkin_type', title='Repartition par type de checkin')
fig.show()

In [ ]:
fig = px.pie(df_delay, names='state', title='Repartition par statut')
fig.show()

## 1.4 Analyse des retards au checkout

In [ ]:
delays = ended['delay_at_checkout_in_minutes'].dropna()
late = delays[delays > 0]
on_time = delays[delays <= 0]

print(f"Locations avec delay renseigne: {len(delays)}")
print(f"En retard (>0 min): {len(late)} ({len(late)/len(delays)*100:.1f}%)")
print(f"A l'heure ou en avance: {len(on_time)} ({len(on_time)/len(delays)*100:.1f}%)")
print(f"Retard moyen (quand retard): {late.mean():.0f} min ({late.mean()/60:.1f}h)")
print(f"Retard median (quand retard): {late.median():.0f} min")

In [ ]:
# Distribution des retards (clippe pour lisibilite)
fig = px.histogram(
    delays.clip(-200, 500), 
    nbins=100, 
    title='Distribution des retards au checkout (clippe -200 a 500 min)',
    labels={'value': 'Retard (minutes)', 'count': 'Nombre'}
)
fig.add_vline(x=0, line_dash='dash', line_color='red', annotation_text='Heure prevue')
fig.show()

In [ ]:
# Retards par type de checkin
delay_by_type = ended.groupby('checkin_type')['delay_at_checkout_in_minutes'].agg(['mean', 'median', 'count'])
delay_by_type.columns = ['Retard moyen', 'Retard median', 'Nb locations']
delay_by_type

In [ ]:
fig = px.box(
    ended, x='checkin_type', y='delay_at_checkout_in_minutes',
    title='Distribution des retards par type de checkin',
    range_y=[-200, 500]
)
fig.show()

## 1.5 Impact sur la location suivante

In [ ]:
# Locations qui ont une location precedente (potentiel conflit)
with_prev = df_delay[df_delay['previous_ended_rental_id'].notna()].copy()
print(f"Locations avec location precedente: {len(with_prev)} ({len(with_prev)/total*100:.1f}%)")

# Recuperer le retard de la location precedente
prev_delays = df_delay.set_index('rental_id')['delay_at_checkout_in_minutes']
with_prev['prev_delay'] = with_prev['previous_ended_rental_id'].map(prev_delays)

In [ ]:
# Cas problematiques : retard precedent > buffer disponible
problematic = with_prev[with_prev['prev_delay'] > with_prev['time_delta_with_previous_rental_in_minutes']]
print(f"Cas problematiques (retard depasse le buffer): {len(problematic)}")
print(f"Soit {len(problematic)/len(with_prev)*100:.1f}% des locations consecutives")

# Annulations parmi les cas problematiques
prob_canceled = problematic[problematic['state'] == 'canceled']
print(f"Dont annulations: {len(prob_canceled)} ({len(prob_canceled)/len(problematic)*100:.1f}%)")

In [ ]:
fig = px.histogram(
    with_prev['time_delta_with_previous_rental_in_minutes'],
    nbins=50,
    title='Distribution du delta entre locations consecutives',
    labels={'value': 'Delta (minutes)', 'count': 'Nombre'}
)
fig.show()

## 1.6 Simulation des seuils (threshold)

In [ ]:
results = []
for threshold in [0, 15, 30, 60, 120, 180, 240, 360, 720]:
    for scope in ['all', 'connect']:
        subset = with_prev if scope == 'all' else with_prev[with_prev['checkin_type'] == 'connect']
        prob_sub = problematic if scope == 'all' else problematic[problematic['checkin_type'] == 'connect']
        
        if threshold == 0:
            affected = 0
            solved = 0
        else:
            affected = (subset['time_delta_with_previous_rental_in_minutes'] < threshold).sum()
            solved = (prob_sub['time_delta_with_previous_rental_in_minutes'] + threshold >= prob_sub['prev_delay']).sum()
        
        results.append({
            'threshold_min': threshold,
            'scope': scope,
            'locations_affectees': affected,
            'pct_affectees': round(affected / len(subset) * 100, 1) if len(subset) > 0 else 0,
            'cas_resolus': solved,
            'pct_resolus': round(solved / len(prob_sub) * 100, 1) if len(prob_sub) > 0 else 0
        })

df_sim = pd.DataFrame(results)
df_sim

In [ ]:
# Tradeoff: cas resolus vs locations bloquees
fig = make_subplots(specs=[[{'secondary_y': True}]])

for scope in ['all', 'connect']:
    sub = df_sim[df_sim['scope'] == scope]
    fig.add_trace(
        go.Scatter(x=sub['threshold_min'], y=sub['pct_resolus'], 
                   name=f'Cas resolus ({scope})', mode='lines+markers'),
        secondary_y=False
    )
    fig.add_trace(
        go.Scatter(x=sub['threshold_min'], y=sub['pct_affectees'], 
                   name=f'Locations bloquees ({scope})', mode='lines+markers', line=dict(dash='dash')),
        secondary_y=True
    )

fig.update_layout(title='Tradeoff: cas resolus vs locations bloquees selon le seuil')
fig.update_xaxes(title_text='Seuil (minutes)')
fig.update_yaxes(title_text='% cas problematiques resolus', secondary_y=False)
fig.update_yaxes(title_text='% locations bloquees', secondary_y=True)
fig.show()

## 1.7 Recommandation

**Seuil recommande : 120 minutes** (scope : connect uniquement)

- Resout ~84% des cas problematiques pour les voitures Connect
- Ne bloque que ~36% des locations consecutives Connect
- Les voitures Connect representent 20% du parc mais ont un checkin sans contact, donc plus sensibles aux retards
- Un seuil de 120 min est un bon compromis entre protection des utilisateurs et perte de revenus

---
# Partie 2 : Pricing - Machine Learning

## 2.1 Chargement et exploration

In [ ]:
df_price = pd.read_csv('data/get_around_pricing_project.csv', index_col=0)
df_price.shape

In [ ]:
df_price.head()

In [ ]:
df_price.info()

In [ ]:
df_price.describe()

In [ ]:
df_price.isnull().sum()

## 2.2 Analyse exploratoire

In [ ]:
fig = px.histogram(df_price, x='rental_price_per_day', nbins=50, title='Distribution du prix journalier')
fig.show()

In [ ]:
fig = px.box(df_price, x='car_type', y='rental_price_per_day', title='Prix par type de vehicule')
fig.show()

In [ ]:
fig = px.scatter(df_price, x='engine_power', y='rental_price_per_day', 
                color='fuel', title='Prix vs puissance moteur')
fig.show()

In [ ]:
fig = px.scatter(df_price, x='mileage', y='rental_price_per_day',
                color='car_type', title='Prix vs kilometrage')
fig.show()

In [ ]:
# Correlation des features numeriques et booleennes
bool_cols = df_price.select_dtypes(include='bool').columns
df_corr = df_price.copy()
for col in bool_cols:
    df_corr[col] = df_corr[col].astype(int)
    
corr = df_corr.select_dtypes(include=[np.number]).corr()['rental_price_per_day'].drop('rental_price_per_day').sort_values(ascending=False)
print('Correlations avec le prix:')
print(corr)

In [ ]:
fig = px.bar(
    x=corr.values, y=corr.index, orientation='h',
    title='Correlation des features avec le prix',
    labels={'x': 'Correlation', 'y': 'Feature'}
)
fig.show()

## 2.3 Preprocessing et modelisation

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import joblib

In [ ]:
target = 'rental_price_per_day'
X = df_price.drop(columns=[target])
y = df_price[target]

cat_cols = [c for c in X.columns if X[c].dtype.name in ('object', 'str', 'string', 'string[python]')]
bool_cols = [c for c in X.columns if X[c].dtype.name == 'bool']
num_cols = [c for c in X.columns if c not in cat_cols and c not in bool_cols]

for col in bool_cols:
    X[col] = X[col].astype(int)

print(f'Categoriques: {cat_cols}')
print(f'Booleens: {bool_cols}')
print(f'Numeriques: {num_cols}')

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), cat_cols),
        ('bool', 'passthrough', bool_cols)
    ]
)

In [ ]:
# Linear Regression
lr_pipe = Pipeline([('preprocessor', preprocessor), ('model', LinearRegression())])
lr_pipe.fit(X_train, y_train)
lr_pred = lr_pipe.predict(X_test)

print("=== Linear Regression ===")
print(f"R2:   {r2_score(y_test, lr_pred):.4f}")
print(f"MAE:  {mean_absolute_error(y_test, lr_pred):.2f} EUR")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, lr_pred)):.2f} EUR")

In [ ]:
# Gradient Boosting
gb_pipe = Pipeline([
    ('preprocessor', preprocessor), 
    ('model', GradientBoostingRegressor(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42))
])
gb_pipe.fit(X_train, y_train)
gb_pred = gb_pipe.predict(X_test)

print("=== Gradient Boosting ===")
print(f"R2:   {r2_score(y_test, gb_pred):.4f}")
print(f"MAE:  {mean_absolute_error(y_test, gb_pred):.2f} EUR")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, gb_pred)):.2f} EUR")

In [ ]:
# Cross-validation
cv_scores = cross_val_score(gb_pipe, X, y, cv=5, scoring='r2')
print(f'CV R2: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})')
print(f'Scores par fold: {[round(s, 4) for s in cv_scores]}')

## 2.4 MLflow tracking

In [ ]:
import mlflow
import mlflow.sklearn

mlflow.set_experiment("getaround_pricing")

# Log du modele Linear Regression
with mlflow.start_run(run_name="linear_regression"):
    lr_pipe_mlf = Pipeline([('preprocessor', preprocessor), ('model', LinearRegression())])
    lr_pipe_mlf.fit(X_train, y_train)
    lr_pred_mlf = lr_pipe_mlf.predict(X_test)
    
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_param("n_features", X_train.shape[1])
    mlflow.log_metric("r2", r2_score(y_test, lr_pred_mlf))
    mlflow.log_metric("mae", mean_absolute_error(y_test, lr_pred_mlf))
    mlflow.log_metric("rmse", np.sqrt(mean_squared_error(y_test, lr_pred_mlf)))
    mlflow.sklearn.log_model(lr_pipe_mlf, "model")

# Log du modele Gradient Boosting
with mlflow.start_run(run_name="gradient_boosting"):
    gb_pipe_mlf = Pipeline([
        ('preprocessor', preprocessor),
        ('model', GradientBoostingRegressor(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42))
    ])
    gb_pipe_mlf.fit(X_train, y_train)
    gb_pred_mlf = gb_pipe_mlf.predict(X_test)
    
    mlflow.log_param("model_type", "GradientBoosting")
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("max_depth", 5)
    mlflow.log_param("learning_rate", 0.1)
    mlflow.log_metric("r2", r2_score(y_test, gb_pred_mlf))
    mlflow.log_metric("mae", mean_absolute_error(y_test, gb_pred_mlf))
    mlflow.log_metric("rmse", np.sqrt(mean_squared_error(y_test, gb_pred_mlf)))
    mlflow.sklearn.log_model(gb_pipe_mlf, "model")

print("Runs MLflow enregistres")

## 2.5 Feature importance

In [ ]:
feature_names = (num_cols + 
    list(gb_pipe.named_steps['preprocessor'].named_transformers_['cat'].get_feature_names_out(cat_cols).tolist()) +
    bool_cols)

importances = gb_pipe.named_steps['model'].feature_importances_
fi = pd.Series(importances, index=feature_names).sort_values(ascending=True)

fig = px.bar(x=fi.values, y=fi.index, orientation='h', 
             title='Feature importance (Gradient Boosting)',
             labels={'x': 'Importance', 'y': 'Feature'})
fig.update_layout(height=600)
fig.show()

In [ ]:
# Predictions vs valeurs reelles
fig = px.scatter(x=y_test, y=gb_pred, 
                 labels={'x': 'Prix reel (EUR)', 'y': 'Prix predit (EUR)'},
                 title='Predictions vs valeurs reelles (Gradient Boosting)')
fig.add_trace(go.Scatter(x=[y_test.min(), y_test.max()], y=[y_test.min(), y_test.max()],
                         mode='lines', line=dict(color='red', dash='dash'), name='y=x'))
fig.show()

## 2.6 Export du modele

In [ ]:
joblib.dump(gb_pipe, 'api/model.joblib')
print('Modele exporte dans api/model.joblib')

In [ ]:
# Verification
loaded_model = joblib.load('api/model.joblib')
sample = X_test.iloc[:3]
preds = loaded_model.predict(sample)
print("Test de prediction:")
for i in range(3):
    print(f"  Predit: {preds[i]:.0f} EUR | Reel: {y_test.iloc[i]} EUR")

---
## Synthese

**Analyse des retards :**
- 57.5% des locations se terminent en retard
- 218 cas problematiques identifies sur 1841 locations consecutives (11.8%)
- Un seuil de 120 min sur les voitures Connect resout 84% des conflits en bloquant 36% des locations consecutives Connect

**Pricing ML :**
- Gradient Boosting : R2=0.75, MAE=10.29 EUR
- Features les plus importantes : puissance moteur, kilometrage, GPS, Connect
- Modele deploye via API FastAPI sur HuggingFace Spaces